In [ ]:
using Pkg
repository = @__DIR__
while !isfile(joinpath(repository, "src", "Surge.jl"))
    parent = dirname(repository)
    parent == repository && error("Open this notebook from within the repository.")
    global repository = parent
end
Pkg.activate(joinpath(repository, "Starting_networks"))
include(joinpath(repository, "src", "Surge.jl"))
import .Surge
include(joinpath(repository, "lora_rank_optimisation", "rank_selection.jl"))

In [ ]:
dataset_file = joinpath(repository,"Data","thesis_v4","uniform.jld2")
pretrained_file = joinpath(repository,"Trained_models","drag_source_pretrained.bson")
output_directory = joinpath(repository,"results", "notebooks", "lora_rank_selection")
fold_count = 5
folds = collect(1:5)
thresholds = [0.80,0.90,0.95,0.99]
inner_split_settings = (repeats=collect(1:3),training_event_count=2,validation_event_count=2,
    holdout_seed=62000,selection_seed=63000,training_seed=640000)
settings = (output_directory=output_directory,dataset_file=dataset_file,pretrained_file=pretrained_file,
    measurement_gap=16,training_steps=1024,stations=[2,4,97,99],observation_noise_std=0.0,
    training_noise_seed_offset=710000,validation_noise_seed_offset=720000,optimizer_seed_offset=10000,
    optimizer=(learning_rate=3e-6,betas=(0.9,0.999),epsilon=1e-8))
pilot_training = (
    epochs=10, batch_size=4, rollout_horizons=[16,32,64,128], window_stride=128,
    loss_mode=:balanced, endpoint_horizon=128, normalize_loss=true,
    curriculum_enabled=false, curriculum_boundaries=[0.25,0.375,0.5],
    detach_gradients=false, gradient_steps=16,
    validation_interval=5, patience=0, cache_refresh_interval=25,
)
candidate_training = (
    epochs=50, batch_size=4, rollout_horizons=[16,32,64,128], window_stride=128,
    loss_mode=:balanced, endpoint_horizon=128, normalize_loss=true,
    curriculum_enabled=false, curriculum_boundaries=[0.25,0.375,0.5],
    detach_gradients=false, gradient_steps=16,
    validation_interval=1, patience=10, cache_refresh_interval=25,
)
nothing

In [ ]:
dataset = Surge.load_burgers_ensemble(basename(dataset_file);
    data_path=dirname(dataset_file), relative_times=true)
pretrained_model = Surge.load_model(pretrained_file)
outer_splits = Surge.make_event_splits(length(dataset.solution);fold_count=fold_count,folds=folds,repeats=[1],
    training_event_count=2,validation_event_count=4,selection_seed=42000,training_seed=420000)
nothing

In [ ]:
selected_ranks = LoRARankOptimisation.select_ranks(pretrained_model,dataset,outer_splits;
    settings=settings,pilot_training=pilot_training,candidate_training=candidate_training,
    thresholds=thresholds,inner_split_settings=inner_split_settings)
for fold in folds
    println("Fold ",fold)
    for (layer,rank) in enumerate(selected_ranks[fold])
        println("  Convolution ",layer,": ",rank)
    end
end